# Importing all functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# creating incremental flag

In [0]:
dbutils.widgets.text("incremental_flag",'0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')

# connection to ADLS

In [0]:
spark.conf.set("fs.azure.account.key.saleessa.dfs.core.windows.net","acces_key")

# defining paths

In [0]:
silver_path = "abfss://silver@saleessa.dfs.core.windows.net/"
gold_path = "abfss://gold@saleessa.dfs.core.windows.net/"

# selecting product data

In [0]:
df_src = spark.sql(
    ''' select distinct(sales_Representative) from 
        parquet.`abfss://silver@saleessa.dfs.core.windows.net/`
    
    '''
)

# creating sink dataset

In [0]:
from delta.tables import DeltaTable

path =  "abfss://gold@saleessa.dfs.core.windows.net/dim_representative"

if DeltaTable.isDeltaTable(spark,path):
    df_sink = spark.read.format('delta').load(path).select("representative_key","sales_Representative")
else:
    df_sink = spark.createDataFrame([], "representative_key int,sales_Representative string")
 


In [0]:
df = df_src.join(df_sink,on='sales_Representative',how='left').select(df_src.sales_Representative,df_sink.representative_key)


sales_Representative,representative_key
John Cross,null
Benjamin Murphy,null
Christian Wells DVM,null
Catherine Weaver,null
Adam Williams,null
Katherine Ross,null
Brandon Rodriguez,null
Gregory Clark,null
Eric Gilmore,null
Veronica Ortiz,null


# old and new records

In [0]:
df_old = df.filter(df.representative_key.isNotNull())
df_new = df.filter(df.representative_key.isNull())

In [0]:
if incremental_flag=='0':
    max_value = 1
else:
    max_value = df_old.select(max(df_old.representative_key)).collect()[0][0]

max_value

1

# adding serogate keys to new data

In [0]:
df_new = df_new.withColumn('representative_key',max_value + monotonically_increasing_id())

In [0]:
union_df = df_old.union(df_new)

In [0]:
path = "abfss://gold@saleessa.dfs.core.windows.net/dim_representative"


if DeltaTable.isDeltaTable(spark,path):
    deltatable = DeltaTable.forPath(spark,path)
    deltatable.alias('target').merge(union_df.alias('source'),'target.representative_key = source.representative_key').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('file upserted')

else:
    union_df.write.format('delta').mode('overwrite').option('mergeSchema','true').save(path)
    print('file created')

file created
